In [20]:
import os
import json
import requests

from google.cloud import discoveryengine_v1 as discoveryengine
from google.api_core.client_options import ClientOptions
from time import time

def load_vertex_config():
    project = os.getenv('VERTEX_PROJECT_NUMBER')
    location = os.getenv('VERTEX_LOCATION', 'global')
    engine_id = os.getenv('VERTEX_ENGINE_ID')
    data_store_id = os.getenv('VERTEX_DATA_STORE_ID')
    language_code = os.getenv('VERTEX_LANGUAGE_CODE', 'it')

    if not project or not engine_id or not data_store_id:
        config_path = os.path.join(os.getcwd(), 'API_keys.json')
        if os.path.exists(config_path):
            try:
                with open(config_path) as f:
                    cfg = json.load(f).get('vertex_ai', {})
                project = project or cfg.get('project_number')
                location = cfg.get('location', location)
                engine_id = engine_id or cfg.get('engine_id')
                data_store_id = data_store_id or cfg.get('data_store_id')
                language_code = cfg.get('language_code', language_code)
            except (OSError, ValueError) as e:
                print(f"[Vertex Config WARN] Could not load API_keys.json: {e}")

    return project, location, engine_id, data_store_id, language_code

def get_search_client(location):
    client_options = None
    if location and location != "global":
        client_options = ClientOptions(
            api_endpoint=f"{location}-discoveryengine.googleapis.com"
        )
    return discoveryengine.SearchServiceClient(client_options=client_options)

def get_completion_client(location):
    client_options = None
    if location and location != "global":
        client_options = ClientOptions(
            api_endpoint=f"{location}-discoveryengine.googleapis.com"
        )
    return discoveryengine.CompletionServiceClient(client_options=client_options)

VERTEX_MAX_RESULTS=100

def vertex_search(query):
    config = load_vertex_config()
    project, location, engine_id, data_store_id, language_code = config

    if not project or not engine_id:
        print("[Vertex Search ERROR] Missing project_number or engine_id in config")
        return [], 0

    client = get_search_client(location)

    serving_config = (
        f"projects/{project}/locations/{location}"
        f"/collections/default_collection/engines/{engine_id}"
        f"/servingConfigs/default_search"
    )

    try:
        request = discoveryengine.SearchRequest(
            serving_config=serving_config,
            query=query,
            page_size=VERTEX_MAX_RESULTS,
            offset=0,
            query_expansion_spec=discoveryengine.SearchRequest.QueryExpansionSpec(
                condition=discoveryengine.SearchRequest.QueryExpansionSpec.Condition.AUTO,
            ),
            spell_correction_spec=discoveryengine.SearchRequest.SpellCorrectionSpec(
                mode=discoveryengine.SearchRequest.SpellCorrectionSpec.Mode.AUTO,
            ),
            language_code=language_code,
        )
        response = client.search(request)
        return (response)
    except Exception as e:
        print(f"[Vertex Search ERROR] {e}")
        return [], 0

def vertex_autocomplete(query, max_suggestions=5):
    config = load_vertex_config()
    print(config)
    project, location, engine_id, data_store_id, language_code = config

    if not project or not data_store_id:
        print("project or data_store_id not found")
        return []

    client = get_completion_client(location)

    data_store_path = (
        f"projects/{project}/locations/{location}"
        f"/collections/default_collection/dataStores/{data_store_id}"
    )

    try:
        request = discoveryengine.CompleteQueryRequest(
            data_store=data_store_path,
            query=query,
            query_model="document-completable",
            include_tail_suggestions=True,
        )
        response = client.complete_query(request)
        return [s.suggestion for s in response.query_suggestions][:max_suggestions]
    except Exception as e:
        print(f"[Vertex Autocomplete ERROR] {e}")
        return []

In [21]:
vertex_search("porto")

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.